In [8]:
import pandas as pd
import numpy as np

rt_df = pd.read_csv("data/SPRT_LogLin_216.csv")

gpt2 = pd.read_csv("data_output/bk21_with_gpt2.csv")
g270 = pd.read_csv("data_output/bk21_with_gemma270m.csv")
g12b = pd.read_csv("data_output/bk21_with_gemma12b.csv")

gpt2_cols = ['ITEM', 'condition', 'gpt2_uni_surprisal', 'gpt2_bi_surprisal', 'cloze_surprisal']
g270_cols = ['ITEM', 'condition', 'gemma270m_uni_surprisal', 'gemma270m_bi_surprisal', 'cloze_surprisal']
g12b_cols = ['ITEM', 'condition', 'gemma12b_uni_surprisal', 'gemma12b_bi_surprisal', 'cloze_surprisal']

In [9]:
merged = pd.merge(rt_df, gpt2[gpt2_cols], on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g270[g270_cols], on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g12b[g12b_cols], on=['ITEM', 'condition'], how='left')

# control variables
# A. Word Length (Characters)
merged['word_length'] = merged['critical_word'].str.len()

# B. Word Position 
merged['word_position'] = merged['position']

# C. Predictor Transformation: Squared Surprisal
for model in ['gpt2', 'gemma270m', 'gemma12b']:
    merged[f'{model}_uni_sq'] = merged[f'{model}_uni_surprisal']**2
    merged[f'{model}_bi_sq'] = merged[f'{model}_bi_surprisal']**2

merged['cloze_surp_sq'] = merged['cloze_surprisal']**2
merged['log_rt'] = np.log(merged['SUM_3RT'])

merged = merged.dropna(subset=['gpt2_uni_surprisal', 'SUM_3RT', 'cloze_surprisal'])

merged.to_csv("data_output/master_modeling_data.csv", index=False)

print(f"Merge Complete! Total rows: {len(merged)}")

Merge Complete! Total rows: 46092
